# Chapter 4: Comparison & Correlation
## Python Exercise — App Store Data

---

**Dataset:** 300 apps from an app store survey  
**Our question:** *Do apps with more users tend to have higher ratings?*

This notebook is in **two parts**:

| Part | Description | Style |
|------|-------------|-------|
| **Part 1** | Worked examples — read, run, understand | Fully guided |
| **Part 2** | Try it yourself — use AI to help you | Independent + AI-assisted |

---

### Variables in the dataset

| Column | Description |
|--------|-------------|
| `app_id` | Unique app identifier |
| `category` | App category (Social Media, Gaming, Productivity, Health & Fitness, Education) |
| `monthly_active_users` | Number of monthly active users (MAU) |
| `ln_mau` | Natural log of monthly active users |
| `app_rating` | Average star rating (1–5) |
| `num_reviews` | Total number of user reviews |
| `price` | App price in £ (0 = free) |
| `is_free` | True if the app is free |
| `months_since_update` | How many months since the last update |
| `size_group` | Binned size: Small / Medium / Large |

---
# 🟢 PART 1 — Worked Examples
### Read each cell carefully, then run it. The explanation comes *before* the code.
---

## Step 1: Load Libraries and Data

Before we can do anything, we need to **import** the tools we'll use and **load** our dataset.

- `pandas` — for organising data in tables
- `matplotlib` and `seaborn` — for charts
- `numpy` — for maths operations

We then use `pd.read_csv()` to load the data from the CSV file into a **DataFrame** — think of it like an Excel spreadsheet inside Python.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Style settings — keeps charts looking clean
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Load the dataset
df = pd.read_csv('app_store_data.csv')

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## Step 2: Explore the Data

Before plotting anything, always **look at the numbers** first. The `.describe()` method gives us summary statistics: mean, standard deviation, min, max, and quartiles.

Pay attention to:
- The **range** of `monthly_active_users` — is it symmetric or skewed?
- The **mean vs median** for `app_rating` — are they similar?

In [ ]:
# Summary statistics for our two key variables
df[['app_rating', 'monthly_active_users', 'ln_mau']].describe().round(2)

In [ ]:
# How many apps per category?
print('Apps by category:')
print(df['category'].value_counts())

print('\nApps by size group:')
print(df['size_group'].value_counts())

## Step 3: Visualise the Distributions

Before comparing variables, look at each one on its own. This is called examining the **marginal distribution**.

We'll plot:
1. `app_rating` — a histogram
2. `monthly_active_users` — a histogram (expect a long right tail!)

A **long right tail** means most values are small, but a few are very large. This is common for counts (users, sales, followers).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Plot 1: App Rating ---
axes[0].hist(df['app_rating'], bins=20, color='#0D9488', edgecolor='white', linewidth=0.6)
axes[0].set_title('Distribution of App Rating (y)')
axes[0].set_xlabel('App Rating (1–5 stars)')
axes[0].set_ylabel('Number of Apps')
axes[0].axvline(df['app_rating'].mean(), color='#F59E0B', linestyle='--', linewidth=1.5, label=f'Mean = {df["app_rating"].mean():.2f}')
axes[0].axvline(df['app_rating'].median(), color='#DC2626', linestyle=':', linewidth=1.5, label=f'Median = {df["app_rating"].median():.2f}')
axes[0].legend(fontsize=9)

# --- Plot 2: Monthly Active Users (raw) ---
axes[1].hist(df['monthly_active_users'], bins=30, color='#1B2A4A', edgecolor='white', linewidth=0.6)
axes[1].set_title('Distribution of Users (raw) — x')
axes[1].set_xlabel('Monthly Active Users')
axes[1].set_ylabel('Number of Apps')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1e6)}M' if x >= 1e6 else f'{int(x/1e3)}K'))

# --- Plot 3: Log of Monthly Active Users ---
axes[2].hist(df['ln_mau'], bins=25, color='#475569', edgecolor='white', linewidth=0.6)
axes[2].set_title('Distribution of ln(Users) — log-transformed x')
axes[2].set_xlabel('ln(Monthly Active Users)')
axes[2].set_ylabel('Number of Apps')

plt.suptitle('Step 3: Marginal Distributions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n🔎 Notice: The raw user count is highly skewed (long right tail).')
print('   After taking the log, the distribution is much more symmetric.')
print('   This is exactly why we use log transformations for variables like user counts!')

## Step 4: Conditional Means — E[y | x]

This is the heart of Chapter 4. We divide apps into **size groups** (small, medium, large) and ask:
*"What is the average rating within each group?"*

In the notation from the slides: **E[y | x]** — the expected value of y, given x.

We use `.groupby()` to split the data by group, then `.agg()` to calculate statistics within each group.

In [ ]:
# Define a clean order for the size groups
size_order = ['Small\n(<10K users)', 'Medium\n(10K–100K)', 'Large\n(>100K)']

# Compute conditional statistics: mean, median, std, count
cond_stats = (
    df.groupby('size_group', observed=True)['app_rating']
    .agg(['mean', 'median', 'std', 'count'])
    .reindex(size_order)  # enforce display order
    .round(3)
)
cond_stats.columns = ['Mean Rating E[y|x]', 'Median', 'Std Dev', 'N apps']
print('Conditional statistics of App Rating by Size Group:')
print(cond_stats)

print('\n💡 Interpretation: Mean rating rises from small to large apps.')
print('   This is positive mean-dependence: E[y|x] varies with x.')

In [ ]:
# Bar chart of conditional means
fig, ax = plt.subplots(figsize=(8, 4.5))

means = cond_stats['Mean Rating E[y|x]']
colors = ['#BFDBFE', '#0D9488', '#1B2A4A']

bars = ax.bar(range(len(size_order)), means.values, color=colors, width=0.55, edgecolor='white')

# Add value labels on bars
for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f} ★', ha='center', va='bottom', fontsize=12, fontweight='bold', color='#1B2A4A')

ax.set_xticks(range(len(size_order)))
ax.set_xticklabels(['Small\n(<10K users)', 'Medium\n(10K–100K)', 'Large\n(>100K)'], fontsize=11)
ax.set_ylabel('Mean App Rating  E[y | x]')
ax.set_title('Conditional Mean Rating by App Size Group', fontweight='bold')
ax.set_ylim(2.5, 4.2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

plt.tight_layout()
plt.show()

## Step 5: Scatterplot — Joint Distribution

A **scatterplot** shows us both variables at the same time — every dot is one app.

We'll make two versions:
- **Left:** raw users on the x-axis — expect it to be squashed on the left
- **Right:** log(users) on the x-axis — the pattern becomes much clearer

This is why log transformations are so useful: they spread out the crowded area and compress the outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: raw scale ---
axes[0].scatter(
    df['monthly_active_users'], df['app_rating'],
    alpha=0.35, s=22, color='#0D9488', edgecolors='none'
)
axes[0].set_xlabel('Monthly Active Users')
axes[0].set_ylabel('App Rating')
axes[0].set_title('Scatterplot (raw scale)', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{int(x/1e6)}M' if x >= 1e6 else f'{int(x/1e3)}K'
))

# --- Right: log scale ---
axes[1].scatter(
    df['ln_mau'], df['app_rating'],
    alpha=0.35, s=22, color='#1B2A4A', edgecolors='none'
)
# Add a trend line using numpy
m, b = np.polyfit(df['ln_mau'], df['app_rating'], 1)  # linear fit
x_line = np.linspace(df['ln_mau'].min(), df['ln_mau'].max(), 100)
axes[1].plot(x_line, m * x_line + b, color='#F59E0B', linewidth=2, label=f'Trend line')
axes[1].set_xlabel('ln(Monthly Active Users)')
axes[1].set_ylabel('App Rating')
axes[1].set_title('Scatterplot (log scale)', fontweight='bold')
axes[1].legend()

plt.suptitle('Step 5: Joint Distribution — Rating vs Users', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔎 Notice: On the raw scale, most dots crowd the left side — hard to see a pattern.')
print('   On the log scale, the upward trend is clearly visible.')

## Step 6: Bin-Scatter

A **bin-scatter** is a cleaner way to see the conditional mean pattern when there are many observations.

**How it works:**
1. Sort all apps by `ln_mau` (from fewest to most users)
2. Divide them into equal-sized bins (e.g., 10 groups of 30 apps each)
3. For each bin, plot the **mean x** against the **mean y**

The result is a clean picture of the trend without the noise of individual dots.

In [ ]:
n_bins = 10

# Use pd.qcut to split into equal-frequency bins based on ln_mau
df['mau_bin'] = pd.qcut(df['ln_mau'], q=n_bins, labels=False)

# Compute mean x and mean y within each bin
bin_means = df.groupby('mau_bin', observed=True).agg(
    mean_ln_mau=('ln_mau', 'mean'),
    mean_rating=('app_rating', 'mean'),
    n=('app_rating', 'count')
).reset_index()

# Plot the bin-scatter
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(bin_means['mean_ln_mau'], bin_means['mean_rating'],
           s=90, color='#0D9488', zorder=5, edgecolors='#1B2A4A', linewidths=0.8)

# Connect dots with a line to make the trend visible
ax.plot(bin_means['mean_ln_mau'], bin_means['mean_rating'],
        color='#0D9488', alpha=0.4, linewidth=1.5)

ax.set_xlabel('Mean ln(Monthly Active Users) within bin')
ax.set_ylabel('Mean App Rating within bin')
ax.set_title(f'Bin-Scatter: Mean Rating vs Mean ln(Users)\n({n_bins} equal-frequency bins)', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n💡 Each dot represents ~30 apps (1/10 of 300).')
print('   The overall upward trend confirms: more users → higher average rating.')

## Step 7: Box Plot and Violin Plot

The conditional mean tells us about the average — but what about the **spread**? Are the distributions more spread out for small apps?

A **box plot** shows:
- The **median** (middle line)
- The **interquartile range** (box = middle 50% of values)
- **Whiskers** extending to 1.5× IQR
- Individual **outliers** beyond that

A **violin plot** shows the same information plus the full density shape.

In [ ]:
# Clean order for plots
size_order_clean = ['Small\n(<10K users)', 'Medium\n(10K–100K)', 'Large\n(>100K)']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette = ['#BFDBFE', '#0D9488', '#1B2A4A']

# --- Box plot ---
sns.boxplot(
    data=df, x='size_group', y='app_rating',
    order=size_order_clean,
    palette=palette,
    width=0.5, linewidth=1.2,
    ax=axes[0]
)
axes[0].set_title('Box Plot: Rating by Size Group', fontweight='bold')
axes[0].set_xlabel('App Size Group')
axes[0].set_ylabel('App Rating')
axes[0].set_ylim(1, 5.5)

# --- Violin plot ---
sns.violinplot(
    data=df, x='size_group', y='app_rating',
    order=size_order_clean,
    palette=palette,
    inner='quartile',
    ax=axes[1]
)
axes[1].set_title('Violin Plot: Rating by Size Group', fontweight='bold')
axes[1].set_xlabel('App Size Group')
axes[1].set_ylabel('App Rating')

plt.suptitle('Step 7: Conditional Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔎 Notice: The median rises across groups (positive association).')
print('   Small apps show more spread — wider range of ratings.')
print('   Large apps are more tightly clustered around higher values.')

## Step 8: Covariance and Correlation

Now we put a **number** on the association.

**Covariance** tells us the direction (positive or negative) but is hard to interpret because it depends on the units.

**Correlation** fixes this by standardising: it is always between −1 and +1.

We'll compute both manually (to see the formula in action) and then use pandas built-in methods.

In [ ]:
x = df['monthly_active_users']
y = df['app_rating']

# --- Manual calculation (mirrors the formula from the slides) ---
n = len(df)
x_bar = x.mean()
y_bar = y.mean()

# Covariance = average of (xi - x_bar) * (yi - y_bar)
cov_manual = ((x - x_bar) * (y - y_bar)).sum() / n

# Correlation = covariance / (std_x * std_y)
std_x = x.std(ddof=0)   # population std (divides by n, matching the formula)
std_y = y.std(ddof=0)
corr_manual = cov_manual / (std_x * std_y)

print('=== Manual Calculation ===')
print(f'Covariance(MAU, Rating)     = {cov_manual:,.1f}')
print(f'Std Dev of MAU              = {std_x:,.1f}')
print(f'Std Dev of Rating           = {std_y:.4f}')
print(f'Correlation(MAU, Rating)    = {corr_manual:.4f}')

# --- Pandas built-in ---
corr_pandas = x.corr(y)
print(f'\n=== Pandas .corr() result   = {corr_pandas:.4f}  (slight diff: pandas uses sample std)')

print('\n💡 Interpretation:')
print(f'   r ≈ {corr_pandas:.2f} → positive, moderate association.')
print('   Apps with more users tend to be rated higher — but not perfectly so.')

In [ ]:
# Now compare correlations using ln_mau vs raw MAU
corr_raw = df['monthly_active_users'].corr(df['app_rating'])
corr_log = df['ln_mau'].corr(df['app_rating'])

print('=== Correlation Comparison ===')
print(f'Corr(raw MAU,  rating) = {corr_raw:.3f}')
print(f'Corr(ln(MAU),  rating) = {corr_log:.3f}')
print('\n💡 The log transformation reveals a stronger linear pattern (higher r).')
print('   This is because the relationship is more linear on the log scale.')

## Step 9: Correlation by App Category

The overall correlation is 0.51 (log scale). But does this hold across all categories?

We can use `.groupby()` again — this time to compute the correlation separately for each category.

This mirrors the industry-level breakdown in the original Békés-Kézdi textbook.

In [ ]:
# Correlation within each category
cat_corr = (
    df.groupby('category')
    .apply(lambda g: g['ln_mau'].corr(g['app_rating']), include_groups=False)
    .reset_index()
)
cat_corr.columns = ['Category', 'Corr(ln_MAU, Rating)']
cat_corr = cat_corr.sort_values('Corr(ln_MAU, Rating)', ascending=False).reset_index(drop=True)

# Add overall
overall_row = pd.DataFrame([{'Category': '── All Categories ──', 'Corr(ln_MAU, Rating)': corr_log}])
cat_corr = pd.concat([cat_corr, overall_row], ignore_index=True)

print(cat_corr.to_string(index=False))

print('\n💡 Notice how the correlation varies by category!')
print('   A high overall correlation can mask differences between groups.')
print('   Always check subgroups when the overall pattern is interesting.')

In [ ]:
# Bar chart of correlations by category
plot_data = cat_corr[cat_corr['Category'] != '── All Categories ──'].copy()

fig, ax = plt.subplots(figsize=(9, 4.5))

colors_bar = ['#0D9488' if v > 0 else '#DC2626' for v in plot_data['Corr(ln_MAU, Rating)']]
bars = ax.barh(plot_data['Category'], plot_data['Corr(ln_MAU, Rating)'],
               color=colors_bar, edgecolor='white', height=0.55)

# Overall reference line
ax.axvline(corr_log, color='#F59E0B', linestyle='--', linewidth=1.5, label=f'Overall r = {corr_log:.2f}')
ax.axvline(0, color='#94A3B8', linewidth=0.8)

for bar, val in zip(bars, plot_data['Corr(ln_MAU, Rating)']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Correlation coefficient  r')
ax.set_title('Correlation between ln(Users) and Rating by Category', fontweight='bold')
ax.legend()
ax.set_xlim(-0.1, 0.85)

plt.tight_layout()
plt.show()

## Step 10: Part 1 Summary

Let's review what we've done and found:

| Step | What we did | Key finding |
|------|-------------|-------------|
| 1–2 | Loaded and explored data | 300 apps, 5 categories |
| 3 | Plotted distributions | User counts are highly skewed; ratings roughly normal |
| 4 | Computed conditional means E[y\|x] | Larger apps → higher mean rating |
| 5 | Scatterplot | Pattern clearer after log transformation |
| 6 | Bin-scatter | Confirms positive, roughly linear trend |
| 7 | Box/violin plots | Median rises across size groups; small apps more spread |
| 8 | Covariance & correlation | r ≈ 0.51 (log scale) — positive, moderate |
| 9 | By category | Correlation varies: Health & Fitness highest, Gaming lowest |

**Main conclusion:** There is a positive, moderate association between app size and rating (r ≈ 0.51 after log transformation). Larger apps tend to be rated higher, but there is considerable variation — knowing an app's user count does not reliably predict its rating.

---
# 🟡 PART 2 — Try It Yourself

### Instructions

In this part, you will carry out your **own analysis** using the same dataset. You are expected to:

1. **Use an AI assistant** (Claude, ChatGPT, GitHub Copilot, etc.) to help you write and debug code
2. **Understand what the code does** — don't just paste output without explanation
3. **Write your interpretations** in the text cells below each task

### How to use AI effectively

Good AI prompts for this exercise:
> *"Using pandas and seaborn, write Python code to create a bin-scatter of app_rating vs ln_mau for only the Gaming category. Use 8 bins."*

> *"Explain what a correlation of 0.12 between price and app_rating means in plain English."*

> *"My code gives a KeyError on 'size_group'. What might be wrong?"*

---

## Task 1: Replicate the Analysis for One Category

Choose **one app category** and replicate the core Part 1 analysis for that category alone:

- Filter the data to your chosen category
- Compute the conditional means by size group
- Create a scatterplot (log scale) with a trend line
- Compute the correlation between `ln_mau` and `app_rating`

**Chosen category:** *(write your choice here before coding)*

In [ ]:
# 📝 Your code here — use AI to help you write it
# Hint: to filter, use:  df_cat = df[df['category'] == 'Gaming']



**✏️ Your interpretation** *(write 3–5 sentences here)*:

- What is the correlation coefficient for your chosen category?
- Is it higher or lower than the overall r ≈ 0.51?
- What do the conditional means tell you?
- What might explain this pattern?

*[Write here...]*

## Task 2: Compare Two Categories Side by Side

Choose **two categories** and compare them directly:

- Create side-by-side scatterplots (one per category, log scale)
- Add a trend line to each
- Report the correlation for each
- Comment on which category shows a stronger size–rating relationship and why you think that might be

In [ ]:
# 📝 Your code here
# Hint: use fig, axes = plt.subplots(1, 2, figsize=(12, 5)) for side-by-side plots



**✏️ Your interpretation** *(write 3–5 sentences)*:

*[Write here...]*

## Task 3: Explore a Third Variable

So far we've only looked at `ln_mau` and `app_rating`. The dataset has other variables too.

**Your task:** Investigate the relationship between **`months_since_update`** (or **`price`**) and `app_rating`.

Steps to follow:
1. Compute the correlation between your chosen x and `app_rating`
2. Create a scatterplot
3. Compute conditional means by creating your own size bins (hint: use `pd.qcut()`)
4. Interpret the result

> **AI prompt idea:** *"Help me create 4 equal-frequency bins for months_since_update in pandas and compute the mean app_rating for each bin."*

In [ ]:
# 📝 Your code here



**✏️ Your interpretation** *(write 3–5 sentences)*:

- What variable did you choose?
- Is the correlation positive or negative? What does that mean in plain English?
- Is this what you expected? Why or why not?

*[Write here...]*

## Task 4: Correlation Matrix

A **correlation matrix** shows the pairwise correlation between all numeric variables at once.

**Your task:**
1. Create a correlation matrix for the numeric variables in the dataset
2. Visualise it as a heatmap using seaborn
3. Identify the **strongest** and **weakest** correlations with `app_rating`

> **AI prompt idea:** *"Using seaborn, create a correlation heatmap for a pandas dataframe. Show the correlation values in each cell. Use a diverging colour palette centred at zero."*

In [ ]:
# 📝 Your code here
# Hint: first select only numeric columns:
# numeric_cols = df[['app_rating', 'ln_mau', 'num_reviews', 'price', 'months_since_update']]



**✏️ Your interpretation** *(write 3–5 sentences)*:

- Which variable has the strongest positive correlation with `app_rating`?
- Which variable has the weakest (or most negative) correlation?
- Is there any pair of x variables that are strongly correlated with each other? Why might this matter?

*[Write here...]*

## Task 5 (Extension): Reflection

Answer the following questions in plain English (no code needed):

1. **Correlation vs Causation:** We found that larger apps tend to have higher ratings. Does this mean *having more users causes better ratings*? What other explanations could there be?

2. **Zero correlation:** If the correlation between `price` and `app_rating` were 0, would that mean price and rating are completely unrelated? Explain.

3. **Log transformation:** In your own words, explain why we took the natural log of monthly active users. What problem did it solve?

4. **AI reflection:** Which parts of the coding in this exercise did you use AI for? Was the output correct first time, or did you need to adjust the prompts? What did you learn from that process?

**✏️ Your answers:**

**Q1 (Correlation vs Causation):**
*[Write here...]*

**Q2 (Zero correlation):**
*[Write here...]*

**Q3 (Log transformation):**
*[Write here...]*

**Q4 (AI reflection):**
*[Write here...]*

---

### ✅ Submission checklist

Before submitting, make sure:
- [ ] All Part 1 cells have been run (outputs visible)
- [ ] Tasks 1–4 have code AND a written interpretation
- [ ] Task 5 reflection is complete
- [ ] Your notebook is saved as `Ch04_Exercise_[YourName].ipynb`

---
*Exercise for Chapter 4: Comparison & Correlation — based on Békés & Kézdi (2021)*